# Notebook 36｜葡萄牙真实水样 P/B/T 真值验证

本 Notebook 将 **Notebook 24 揭盲前冻结的谱图预测** 与 **Notebook 25 后揭示的葡萄牙化合物身份**、项目内已冻结的实验 P/B/T 标签连接。

验证严格分层：

- `PRIMARY_UNSEEN_COMPOUND`：训练阶段完全排除的 `external_novel` 化合物，可用于未见化合物的真实基质验证；
- `SECONDARY_SEEN_COMPOUND_NEW_MATRIX`：开发集出现过的化合物，只能检验真实污水谱图的测量域迁移；
- `NO_FROZEN_TRUTH`：没有冻结实验真值，只保留身份恢复与风险排序结论。

> Watchlist 身份绝不当作 PBT 真值。由于主验证层各端点仅有 1–7 个真值，本步骤属于小样本支持性案例验证，不构成大样本决定性外部验证。

## 运行方法

1. 内核选择 **Python (env_ms)**；
2. 点击 **Kernel → Restart Kernel and Run All Cells**；
3. 不要修改阈值、标签或训练/外部划分；
4. 最后一格必须显示 `PORTUGAL_REAL_MATRIX_VALIDATION_COMPLETE_UNDERPOWERED` 且全部 QC 为 `PASS`。

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

assert 'env_ms' in sys.executable.lower(), f'请切换到 Python (env_ms) 内核；当前解释器：{sys.executable}'
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs/pbt_msms_v1').exists()), None)
assert ROOT is not None, f'无法从 {cwd} 定位项目根目录'
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbt_msms.portugal_pbt_truth_validation import FINAL_STATUS, TASKS, run

print('Project root:', ROOT)
print('Python:', sys.executable)
print('Expected final status:', FINAL_STATUS)

In [ ]:
manifest = run(ROOT)
OUT = ROOT / 'external_data/portugal_wastewater_5830725/08_external_validation/36_portugal_pbt_truth_validation_v1'
print(json.dumps({
    'status': manifest['status'],
    'qc_all_required_pass': manifest['qc_all_required_pass'],
    'counts': manifest['counts'],
}, ensure_ascii=False, indent=2))

## 真值覆盖与可允许结论

先看每层实际有多少真值，再解释性能；不能把 84 个已鉴定化合物误写成 84 个均具有 P/B/T 真值。

In [ ]:
coverage = pd.read_csv(OUT / '36_truth_coverage.csv')
claim_scope = pd.read_csv(OUT / '36_claim_scope.csv')
display(claim_scope)
display(coverage.pivot(index='task', columns='validation_stratum', values='truth_available_n').fillna(0).astype(int))

## 主验证：训练未见化合物

下表只使用 `external_novel`。`DESCRIPTIVE_UNDERPOWERED` 表示指标可以报告，但必须同时报告样本量，不能据此宣称稳定的总体性能。

In [ ]:
performance = pd.read_csv(OUT / '36_endpoint_performance.csv')
primary = performance.query("evaluation_stratum == 'PRIMARY_UNSEEN_COMPOUND'").copy()
display(primary[['task', 'n_compounds', 'positive_n', 'negative_n', 'metric', 'value', 'ci95_low', 'ci95_high', 'interpretation_status']])

## 辅助验证：已见化合物的新基质谱图

这部分可描述真实污水测量域迁移的一致性，但不能称作未见化合物外部验证。

In [ ]:
secondary = performance.query("evaluation_stratum == 'SECONDARY_SEEN_COMPOUND_NEW_MATRIX'").copy()
display(secondary[['task', 'n_compounds', 'positive_n', 'negative_n', 'metric', 'value', 'ci95_low', 'ci95_high', 'interpretation_status']])

In [ ]:
records = pd.read_csv(OUT / '36_endpoint_validation_records.csv')
primary_records = records.query("validation_stratum == 'PRIMARY_UNSEEN_COMPOUND'")
display(primary_records[['compound_name_published', 'inchikey_2d', 'task', 'truth', 'prediction', 'identified_case_group_n', 'source_datasets']])

In [ ]:
figure_path = OUT / 'figures/36_portugal_pbt_truth_validation.png'
display(Image(filename=str(figure_path), width=1200))

## 冻结状态与 QC

最后一格检查身份唯一性、冻结预测完整性、训练泄漏分层、真值一致性和结论边界。

In [ ]:
qc = pd.read_csv(OUT / '36_validation_qc_checks.csv')
summary = json.loads((OUT / '36_validation_summary.json').read_text(encoding='utf-8'))
display(qc)
assert qc['status'].eq('PASS').all(), qc.loc[~qc['status'].eq('PASS')]
assert manifest['status'] == FINAL_STATUS
assert manifest['qc_all_required_pass'] is True
assert summary['watchlist_used_as_truth'] is False
assert summary['identity_feedback_into_predictions'] is False
print('FINAL STATUS:', manifest['status'])
print('结论:', summary['scientific_conclusion'])
print('输出目录:', OUT)